# Credit Card Fraud Detection — Exploratory Analysis

This notebook explores the real ULB (Université Libre de Bruxelles) credit
card fraud dataset and demonstrates the core methodology. **The full,
authoritative training pipeline lives in `train_model.py`** — run that
script for the complete, reproducible results reported in the README.
This notebook is for exploration and explanation, not the source of truth
for the reported metrics.

Dataset: 284,807 European cardholder transactions from September 2013,
492 confirmed frauds (0.1727%). Download instructions in the README.

## 1. Load and inspect the data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('creditcard.csv')
print(f'Shape: {df.shape}')
print(f'Fraud cases: {df["Class"].sum()} ({100*df["Class"].mean():.4f}%)')
df.head()

## 2. The core challenge: severe class imbalance

Fraud is 0.17% of all transactions. This single fact drives every
methodology decision in this project: how we split the data, which metric
we optimize for, and why accuracy alone is meaningless here.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

df['Class'].value_counts().plot(kind='bar', ax=ax[0], color=['steelblue', 'crimson'])
ax[0].set_title('Class distribution (linear scale)')
ax[0].set_xticklabels(['Legit', 'Fraud'], rotation=0)

df['Class'].value_counts().plot(kind='bar', ax=ax[1], color=['steelblue', 'crimson'], log=True)
ax[1].set_title('Class distribution (log scale — fraud is visible here)')
ax[1].set_xticklabels(['Legit', 'Fraud'], rotation=0)

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=100)
plt.show()

print(f"Imbalance ratio: 1 fraud case per {(df['Class']==0).sum() // df['Class'].sum()} legitimate transactions")

## 3. Why accuracy is a trap on this dataset

A model that predicts "not fraud" for every single transaction would score
99.83% accuracy — while catching zero fraud. This is the single most
important thing to understand before evaluating any model on this data.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split

X = df.drop('Class', axis=1)
y = df['Class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
dummy_acc = dummy.score(X_test, y_test)

print(f"'Always predict not-fraud' baseline accuracy: {dummy_acc:.4f}")
print(f"Fraud cases caught by this baseline: 0 / {y_test.sum()}")
print("\nThis is why every model in this project is evaluated on PR-AUC and")
print("precision/recall, not accuracy alone.")

## 4. Amount and Time — the two features that need scaling

Unlike `V1`-`V28` (already PCA-transformed and roughly standardized),
`Amount` and `Time` are raw. Note: they need **separate** `StandardScaler`
objects — reusing one scaler for both by calling `fit_transform()` twice
silently overwrites the first column's fitted statistics with the second's.
This was a real bug caught and fixed in this project (see `fix_scalers.py`
and the regression test in `tests/test_all.py`).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df[df['Class']==0]['Amount'].plot(kind='hist', bins=50, ax=ax[0], alpha=0.6, label='Legit', color='steelblue')
df[df['Class']==1]['Amount'].plot(kind='hist', bins=50, ax=ax[0], alpha=0.6, label='Fraud', color='crimson')
ax[0].set_title('Transaction Amount distribution')
ax[0].set_xlim(0, 500)
ax[0].legend()

df[df['Class']==0]['Time'].plot(kind='hist', bins=50, ax=ax[1], alpha=0.6, label='Legit', color='steelblue')
df[df['Class']==1]['Time'].plot(kind='hist', bins=50, ax=ax[1], alpha=0.6, label='Fraud', color='crimson')
ax[1].set_title('Time (seconds since first transaction)')
ax[1].legend()

plt.tight_layout()
plt.savefig('amount_time_distribution.png', dpi=100)
plt.show()

print(f"Amount — mean: {df['Amount'].mean():.2f}, median: {df['Amount'].median():.2f}, max: {df['Amount'].max():.2f}")
print(f"Time   — mean: {df['Time'].mean():.2f}, range: 0 to {df['Time'].max():.0f} seconds (~{df['Time'].max()/3600:.1f} hours)")

## 5. Feature correlation with fraud

Since `V1`-`V28` are anonymized, we can't interpret them directly, but we
can rank which ones separate fraud from legitimate transactions most
clearly — this previews the feature importance result from the full model.

In [ ]:
correlations = df.drop('Class', axis=1).corrwith(df['Class']).abs().sort_values(ascending=False)
print("Top 10 features by absolute correlation with fraud:")
print(correlations.head(10))

correlations.head(10).plot(kind='barh', figsize=(8, 5), color='steelblue')
plt.gca().invert_yaxis()
plt.title('Top 10 features correlated with fraud')
plt.tight_layout()
plt.savefig('feature_correlation.png', dpi=100)
plt.show()

## 6. Full pipeline — run separately

The complete, authoritative pipeline — stratified split, correct
per-column scaling, baseline comparison, `class_weight='balanced'` vs.
SMOTE comparison, cross-validation, and final model selection — lives in
**`train_model.py`**. Run it directly:

```bash
python train_model.py
```

This notebook is exploratory; `train_model.py` is the source of truth for
every metric reported in the README.